In [ ]:
import pandas as pd
import numpy as np

PATH = r"BaAM-data-team\merged data sets\aggregated\aggregated_with_gang_indicator.csv"

df = pd.read_csv(PATH)

print("Raw shape:", df.shape)
print("Duplicate (community_area, year):", df.duplicated(["community_area", "year"]).sum())
df.head()


Raw shape: (1063, 30)
Duplicate (community_area, year): 13


,community_area,year,homicide_count,homicide_arrests,homicide_domestic,shooting_count,shooting_arrests,shooting_domestic,pop_total,pop_age_0_17,...,pct_without_hs_diploma,per_capita_income,hardship_index,homicide_arrest_rate,shooting_arrest_rate,homicides_per_10k,shootings_per_10k,is_shooting,gang_activity_score,gang_activity
0,1,2012,0.068182,4.0,0.142857,0.112321,58.0,0.023077,50295.0,8007.0,...,0.300191,23939.0,0.391753,0.666667,0.351515,0.020489,0.011739,1,0.012332,0
1,1,2013,0.045455,1.0,0.142857,0.076923,36.0,0.069231,50295.0,8007.0,...,0.300191,23939.0,0.391753,0.250000,0.318584,0.013659,0.008039,1,0.060300,0
2,1,2014,0.090909,1.0,0.000000,0.074881,32.0,0.030769,50295.0,8007.0,...,0.300191,23939.0,0.391753,0.125000,0.290909,0.027319,0.007826,1,0.106482,0
3,1,2015,0.068182,3.0,0.142857,0.068074,22.0,0.038462,50295.0,8007.0,...,0.300191,23939.0,0.391753,0.500000,0.220000,0.020489,0.007114,1,0.044072,0
4,1,2016,0.056818,3.0,0.000000,0.079646,27.0,0.046154,50295.0,8007.0,...,0.300191,23939.0,0.391753,0.600000,0.230769,0.017074,0.008324,1,0.033715,0


In [21]:
# Aggregate duplicates at (community_area, year)
binary_cols = ["gang_activity", "is_shooting"]

agg_dict = {}
for col in df.columns:
    if col in ["community_area", "year"]:
        continue
    if col in binary_cols:
        agg_dict[col] = "max"
    else:
        agg_dict[col] = "mean"

df = (
    df.groupby(["community_area", "year"], as_index=False)
      .agg(agg_dict)
      .sort_values(["community_area", "year"])
      .reset_index(drop=True)
)

print("After dedup:", df.shape)

# Create target (next year homicide rate)
df["homicides_per_10k_next_year"] = (
    df.groupby("community_area")["homicides_per_10k"].shift(-1)
)

# Create lag1 shooting features
lag_vars = [
    "shooting_count",
    "shootings_per_10k",
    "is_shooting",
    "shooting_arrests",
    "shooting_arrest_rate",
    "shooting_domestic"
]

for col in lag_vars:
    df[f"{col}_lag1"] = df.groupby("community_area")[col].shift(1)

# Drop rows missing target or lag1 values
df = df.dropna(
    subset=["homicides_per_10k_next_year"] + [f"{col}_lag1" for col in lag_vars]
).copy()

print("After target and lag drop:", df.shape)


After dedup: (1050, 30)
After target and lag drop: (900, 37)


In [22]:
# Remove leakage variables
remove_cols = [
    "homicide_count",
    "homicide_arrests",
    "homicide_arrest_rate",
    "homicide_domestic",
    "homicides_per_10k",
    "shooting_count",
    "shootings_per_10k",
    "is_shooting",
    "shooting_arrests",
    "shooting_arrest_rate",
    "shooting_domestic"
]

df = df[[c for c in df.columns if c not in remove_cols]].copy()

target = "homicides_per_10k_next_year"
id_cols = ["community_area", "year"]
binary_cols = ["gang_activity", "is_shooting_lag1"]

# Identify continuous predictors to scale
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in id_cols + [target] + binary_cols]

# Z-score standardization
means = df[scale_cols].mean()
stds = df[scale_cols].std(ddof=0).replace(0, np.nan)

df[scale_cols] = (df[scale_cols] - means) / stds

print("Final shape:", df.shape)


Final shape: (900, 26)


In [ ]:
# Manual lag alignment check
check = df[df["community_area"] == 1][
    ["year", "shooting_count_lag1", "shootings_per_10k_lag1"]
].head(5)

print(check)

# Save final dataset
OUTPUT_PATH = r"BaAM-data-team\Modelling\final_baseline_dataset.csv"
df.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)


   year  shooting_count_lag1  shootings_per_10k_lag1
1  2013            -0.151369               -0.169662
2  2014            -0.375854               -0.295820
3  2015            -0.388805               -0.303098
4  2016            -0.431975               -0.327360
5  2017            -0.358586               -0.286116
Saved to: C:\Users\aaron\OneDrive\文档\GitHub\BaAM-data-team\Modelling\final_baseline_dataset.csv
